In [1]:
from google.colab import files

uploaded = files.upload()
INPUT_FILE = next(iter(uploaded))

print("Uploaded:", INPUT_FILE)

Saving Class_B_X_YouTube_Final.csv to Class_B_X_YouTube_Final.csv
Uploaded: Class_B_X_YouTube_Final.csv


In [2]:
import pandas as pd
import numpy as np
import re
import math
import unicodedata
from collections import Counter
from difflib import SequenceMatcher
from pathlib import Path
from scipy.stats import chi2_contingency, mannwhitneyu, kruskal

OUTPUT_DIR = Path("class_b_statistics")
OUTPUT_DIR.mkdir(exist_ok=True)

df = pd.read_csv(INPUT_FILE, dtype=str)

HANDLE_COL = "handle"
PLATFORM_COL = "platform"
QID_COL = "qid"
LABEL_COL = "itemLabel"

OCCUPATION_COL = next(
    (c for c in ["occupations", "occupation"] if c in df.columns),
    None
)

GENDER_COL = next(
    (c for c in ["gender", "genderLabel"] if c in df.columns),
    None
)

CITIZENSHIP_COL = next(
    (c for c in ["citizenships", "citizenship"] if c in df.columns),
    None
)

LANGUAGE_COL = next(
    (c for c in ["languages", "language"] if c in df.columns),
    None
)

required = [HANDLE_COL, PLATFORM_COL, QID_COL]

missing_required = [
    column for column in required
    if column not in df.columns
]

if missing_required:
    raise ValueError(
        f"Missing required columns: {missing_required}"
    )

print("Original shape:", df.shape)
print("Columns:", list(df.columns))

Original shape: (20000, 12)
Columns: ['item', 'qid', 'platform', 'property_id', 'handle', 'itemLabel', 'occupations', 'gender', 'citizenships', 'languages', 'date_of_birth', 'age']


In [3]:
import pandas as pd
import numpy as np
import re
import math
import unicodedata
from collections import Counter
from difflib import SequenceMatcher
from pathlib import Path
from scipy.stats import chi2_contingency, mannwhitneyu, kruskal

OUTPUT_DIR = Path("class_b_statistics")
OUTPUT_DIR.mkdir(exist_ok=True)

df = pd.read_csv(INPUT_FILE, dtype=str)

HANDLE_COL = "handle"
PLATFORM_COL = "platform"
QID_COL = "qid"
LABEL_COL = "itemLabel"

OCCUPATION_COL = next(
    (c for c in ["occupations", "occupation"] if c in df.columns),
    None
)

GENDER_COL = next(
    (c for c in ["gender", "genderLabel"] if c in df.columns),
    None
)

CITIZENSHIP_COL = next(
    (c for c in ["citizenships", "citizenship"] if c in df.columns),
    None
)

LANGUAGE_COL = next(
    (c for c in ["languages", "language"] if c in df.columns),
    None
)

required = [HANDLE_COL, PLATFORM_COL, QID_COL]

missing_required = [
    column for column in required
    if column not in df.columns
]

if missing_required:
    raise ValueError(
        f"Missing required columns: {missing_required}"
    )

print("Original shape:", df.shape)
print("Columns:", list(df.columns))

Original shape: (20000, 12)
Columns: ['item', 'qid', 'platform', 'property_id', 'handle', 'itemLabel', 'occupations', 'gender', 'citizenships', 'languages', 'date_of_birth', 'age']


In [4]:

for column in df.columns:
    if df[column].dtype == "object":
        df[column] = df[column].str.strip()

df = df.dropna(subset=[HANDLE_COL])
df = df[df[HANDLE_COL].ne("")]

original_rows = len(df)

df = df.drop_duplicates(
    subset=[QID_COL, PLATFORM_COL, HANDLE_COL]
).reset_index(drop=True)

print("Duplicate rows removed:", original_rows - len(df))
print("Analysis rows:", len(df))
print("Unique people:", df[QID_COL].nunique())
print("Unique handles:", df[HANDLE_COL].nunique())
print(df[PLATFORM_COL].value_counts())

Duplicate rows removed: 0
Analysis rows: 20000
Unique people: 19130
Unique handles: 19750
platform
x_twitter         10000
youtube_handle    10000
Name: count, dtype: int64


In [5]:

def segment_username(username):
    text = unicodedata.normalize("NFKC", str(username))

    text = re.sub(r"[_\-.]+", " ", text)
    text = re.sub(r"(?<=[a-z])(?=[A-Z])", " ", text)
    text = re.sub(r"(?<=[A-Z])(?=[A-Z][a-z])", " ", text)
    text = re.sub(r"(?<=[A-Za-z])(?=\d)", " ", text)
    text = re.sub(r"(?<=\d)(?=[A-Za-z])", " ", text)

    return re.sub(r"\s+", " ", text).strip()


def normalize_name(text):
    if pd.isna(text):
        return ""

    text = unicodedata.normalize("NFKD", str(text))
    text = "".join(
        character for character in text
        if not unicodedata.combining(character)
    )

    return re.sub(
        r"[^a-z0-9]",
        "",
        text.lower()
    )


def char_entropy(text):
    text = str(text)

    if not text:
        return 0.0

    counts = Counter(text)
    length = len(text)

    return -sum(
        (count / length) * math.log2(count / length)
        for count in counts.values()
    )


df["handle_clean"] = df[HANDLE_COL].astype(str).str.strip()
df["username_segmented"] = df["handle_clean"].map(
    segment_username
)

handle = df["handle_clean"]

df["length"] = handle.str.len()

df["letter_count"] = handle.map(
    lambda x: sum(character.isalpha() for character in x)
)

df["digit_count"] = handle.map(
    lambda x: sum(character.isdigit() for character in x)
)

df["vowel_count"] = handle.str.lower().map(
    lambda x: sum(character in "aeiou" for character in x)
)

df["underscore_count"] = handle.str.count("_")
df["period_count"] = handle.str.count(r"\.")
df["hyphen_count"] = handle.str.count("-")

df["special_count"] = handle.map(
    lambda x: sum(not character.isalnum() for character in x)
)

df["other_special_count"] = handle.map(
    lambda x: sum(
        not character.isalnum()
        and character not in "_.-"
        for character in x
    )
)

df["vowel_ratio"] = (
    df["vowel_count"]
    / df["letter_count"].replace(0, np.nan)
).fillna(0)

df["char_entropy"] = handle.map(char_entropy)

df["delimiter_piece_count"] = handle.map(
    lambda x: len([
        piece for piece in re.split(r"[_\-.]+", x)
        if piece
    ])
)

In [6]:

df["contains_digit"] = handle.str.contains(
    r"\d", regex=True
).astype(int)

df["ends_with_digit"] = handle.str.contains(
    r"\d$", regex=True
).astype(int)

df["starts_with_letter"] = handle.str.contains(
    r"^[A-Za-z]", regex=True
).astype(int)

df["contains_underscore"] = handle.str.contains(
    "_", regex=False
).astype(int)

df["contains_period"] = handle.str.contains(
    ".", regex=False
).astype(int)

df["contains_hyphen"] = handle.str.contains(
    "-", regex=False
).astype(int)

df["contains_other_special"] = (
    df["other_special_count"] > 0
).astype(int)

df["repeated_char_3plus"] = handle.map(
    lambda x: int(bool(re.search(r"(.)\1{2,}", x)))
)

df["all_lowercase"] = handle.map(
    lambda x: int(
        any(character.isalpha() for character in x)
        and x.islower()
    )
)

df["all_uppercase"] = handle.map(
    lambda x: int(
        any(character.isalpha() for character in x)
        and x.isupper()
    )
)

df["mixed_case"] = handle.map(
    lambda x: int(
        any(character.islower() for character in x)
        and any(character.isupper() for character in x)
    )
)

df["camelcase"] = handle.str.contains(
    r"[a-z][A-Z]",
    regex=True
).astype(int)

df["numeric_only"] = handle.str.fullmatch(
    r"\d+"
).fillna(False).astype(int)

In [7]:
def calculate_name_features(row):
    handle_normalized = normalize_name(
        row["handle_clean"]
    )

    label = row.get(LABEL_COL, "")

    if pd.isna(label) or not str(label).strip():
        return pd.Series({
            "contains_given_name": np.nan,
            "contains_family_name": np.nan,
            "contains_full_name": np.nan,
            "name_match_score": np.nan
        })

    tokens = re.findall(
        r"[A-Za-zÀ-ÿ]+",
        str(label)
    )

    if not tokens:
        return pd.Series({
            "contains_given_name": np.nan,
            "contains_family_name": np.nan,
            "contains_full_name": np.nan,
            "name_match_score": np.nan
        })

    given_name = normalize_name(tokens[0])
    family_name = normalize_name(tokens[-1])
    full_name = normalize_name(label)

    similarities = [
        SequenceMatcher(
            None,
            handle_normalized,
            full_name
        ).ratio()
    ]

    if given_name:
        similarities.append(
            SequenceMatcher(
                None,
                handle_normalized,
                given_name
            ).ratio()
        )

    if family_name:
        similarities.append(
            SequenceMatcher(
                None,
                handle_normalized,
                family_name
            ).ratio()
        )

    return pd.Series({
        "contains_given_name": int(
            bool(given_name)
            and given_name in handle_normalized
        ),
        "contains_family_name": int(
            bool(family_name)
            and family_name in handle_normalized
        ),
        "contains_full_name": int(
            bool(full_name)
            and full_name in handle_normalized
        ),
        "name_match_score": max(similarities)
    })


if LABEL_COL in df.columns:
    name_features = df.apply(
        calculate_name_features,
        axis=1
    )

    df = pd.concat(
        [df, name_features],
        axis=1
    )

In [8]:
NUMERIC_FEATURES = [
    "length",
    "letter_count",
    "digit_count",
    "vowel_count",
    "underscore_count",
    "period_count",
    "hyphen_count",
    "special_count",
    "other_special_count",
    "vowel_ratio",
    "delimiter_piece_count",
    "char_entropy"
]

if "name_match_score" in df.columns:
    NUMERIC_FEATURES.append("name_match_score")

BINARY_FEATURES = [
    "contains_digit",
    "ends_with_digit",
    "starts_with_letter",
    "contains_underscore",
    "contains_period",
    "contains_hyphen",
    "contains_other_special",
    "repeated_char_3plus",
    "all_lowercase",
    "all_uppercase",
    "mixed_case",
    "camelcase",
    "numeric_only"
]

for column in [
    "contains_given_name",
    "contains_family_name",
    "contains_full_name"
]:
    if column in df.columns:
        BINARY_FEATURES.append(column)


numeric_summary = df[NUMERIC_FEATURES].describe(
    percentiles=[0.25, 0.50, 0.75]
).T

numeric_summary = numeric_summary.rename(
    columns={
        "25%": "q1",
        "50%": "median",
        "75%": "q3"
    }
)

numeric_summary.to_csv(
    OUTPUT_DIR / "class_b_overall_numeric_statistics.csv"
)

binary_summary = pd.DataFrame({
    "feature": BINARY_FEATURES,
    "count_positive": [
        df[column].sum()
        for column in BINARY_FEATURES
    ],
    "percentage": [
        df[column].mean() * 100
        for column in BINARY_FEATURES
    ],
    "standard_deviation_pp": [
        df[column].std() * 100
        for column in BINARY_FEATURES
    ]
})

binary_summary.to_csv(
    OUTPUT_DIR / "class_b_overall_binary_statistics.csv",
    index=False
)

display(numeric_summary)
display(binary_summary)

,count,mean,std,min,q1,median,q3,max
length,20000.0,11.510950,3.627704,1.0,9.000000,11.000000,14.000000,48.000000
letter_count,20000.0,10.993500,3.596457,0.0,9.000000,11.000000,13.000000,38.000000
digit_count,20000.0,0.371600,1.074510,0.0,0.000000,0.000000,0.000000,13.000000
vowel_count,20000.0,4.311050,1.860718,0.0,3.000000,4.000000,5.000000,14.000000
underscore_count,20000.0,0.123200,0.394372,0.0,0.000000,0.000000,0.000000,12.000000
period_count,20000.0,0.010100,0.112244,0.0,0.000000,0.000000,0.000000,4.000000
hyphen_count,20000.0,0.011750,0.117952,0.0,0.000000,0.000000,0.000000,3.000000
special_count,20000.0,0.145850,0.433229,0.0,0.000000,0.000000,0.000000,12.000000
other_special_count,20000.0,0.000800,0.089441,0.0,0.000000,0.000000,0.000000,12.000000
vowel_ratio,20000.0,0.386509,0.113611,0.0,0.333333,0.400000,0.454545,1.000000


,feature,count_positive,percentage,standard_deviation_pp
0,contains_digit,2685.0,13.425000,34.092924
1,ends_with_digit,2290.0,11.450000,31.842553
2,starts_with_letter,19528.0,97.640000,15.180314
3,contains_underscore,2135.0,10.675000,30.880286
4,contains_period,180.0,0.900000,9.444282
5,contains_hyphen,215.0,1.075000,10.312599
6,contains_other_special,2.0,0.010000,0.999975
7,repeated_char_3plus,232.0,1.160000,10.707947
8,all_lowercase,14386.0,71.930000,44.935243
9,all_uppercase,217.0,1.085000,10.359930


In [9]:
platform_rows = []

for platform, group in df.groupby(PLATFORM_COL):
    result = {
        "platform": platform,
        "n": len(group),
        "unique_people": group[QID_COL].nunique(),
        "unique_handles": group[HANDLE_COL].nunique()
    }

    for feature in NUMERIC_FEATURES:
        result[f"mean_{feature}"] = group[feature].mean()
        result[f"std_{feature}"] = group[feature].std()
        result[f"median_{feature}"] = group[feature].median()

    for feature in BINARY_FEATURES:
        result[f"pct_{feature}"] = (
            group[feature].mean() * 100
        )

    platform_rows.append(result)

platform_summary = pd.DataFrame(platform_rows)

platform_summary.to_csv(
    OUTPUT_DIR / "class_b_platform_statistics.csv",
    index=False
)

display(platform_summary)

,platform,n,unique_people,unique_handles,mean_length,std_length,median_length,mean_letter_count,std_letter_count,median_letter_count,...,pct_contains_other_special,pct_repeated_char_3plus,pct_all_lowercase,pct_all_uppercase,pct_mixed_case,pct_camelcase,pct_numeric_only,pct_contains_given_name,pct_contains_family_name,pct_contains_full_name
0,x_twitter,10000,9818,9970,10.8032,2.822245,11.0,10.3658,2.931663,11.0,...,0.00,1.17,67.46,1.17,31.15,21.81,0.22,64.954865,68.415246,41.554664
1,youtube_handle,10000,9803,9987,12.2187,4.165910,12.0,11.6212,4.060488,11.0,...,0.02,1.15,76.40,1.00,19.99,15.77,0.08,74.383031,72.476035,60.881093


In [10]:
missing_summary = pd.DataFrame({
    "column": df.columns,
    "missing_count": df.isna().sum().values,
    "missing_percentage": (
        df.isna().mean().values * 100
    )
}).sort_values(
    "missing_percentage",
    ascending=False
)

missing_summary.to_csv(
    OUTPUT_DIR / "class_b_missing_data_summary.csv",
    index=False
)

display(missing_summary.head(20))

,column,missing_count,missing_percentage
9,languages,14551,72.755
11,age,5908,29.540
10,date_of_birth,5837,29.185
8,citizenships,4242,21.210
6,occupations,922,4.610
7,gender,911,4.555
41,contains_full_name,224,1.120
39,contains_given_name,224,1.120
40,contains_family_name,224,1.120
42,name_match_score,224,1.120


In [11]:
missing_summary = pd.DataFrame({
    "column": df.columns,
    "missing_count": df.isna().sum().values,
    "missing_percentage": (
        df.isna().mean().values * 100
    )
}).sort_values(
    "missing_percentage",
    ascending=False
)

missing_summary.to_csv(
    OUTPUT_DIR / "class_b_missing_data_summary.csv",
    index=False
)

display(missing_summary.head(20))

,column,missing_count,missing_percentage
9,languages,14551,72.755
11,age,5908,29.540
10,date_of_birth,5837,29.185
8,citizenships,4242,21.210
6,occupations,922,4.610
7,gender,911,4.555
41,contains_full_name,224,1.120
39,contains_given_name,224,1.120
40,contains_family_name,224,1.120
42,name_match_score,224,1.120


In [12]:
def save_platform_crosstab(column, filename):
    if column is None or column not in df.columns:
        return

    exploded = (
        df[[PLATFORM_COL, column]]
        .dropna()
        .assign(
            category=lambda frame:
                frame[column].str.split(r"\s*\|\s*")
        )
        .explode("category")
    )

    exploded["category"] = exploded["category"].str.strip()
    exploded = exploded[exploded["category"].ne("")]

    table = pd.crosstab(
        exploded[PLATFORM_COL],
        exploded["category"]
    )

    table.to_csv(
        OUTPUT_DIR / filename
    )


save_platform_crosstab(
    GENDER_COL,
    "class_b_platform_by_gender.csv"
)

save_platform_crosstab(
    OCCUPATION_COL,
    "class_b_platform_by_occupation.csv"
)

save_platform_crosstab(
    CITIZENSHIP_COL,
    "class_b_platform_by_citizenship.csv"
)

save_platform_crosstab(
    LANGUAGE_COL,
    "class_b_platform_by_language.csv"
)

In [13]:
def save_platform_crosstab(column, filename):
    if column is None or column not in df.columns:
        return

    exploded = (
        df[[PLATFORM_COL, column]]
        .dropna()
        .assign(
            category=lambda frame:
                frame[column].str.split(r"\s*\|\s*")
        )
        .explode("category")
    )

    exploded["category"] = exploded["category"].str.strip()
    exploded = exploded[exploded["category"].ne("")]

    table = pd.crosstab(
        exploded[PLATFORM_COL],
        exploded["category"]
    )

    table.to_csv(
        OUTPUT_DIR / filename
    )


save_platform_crosstab(
    GENDER_COL,
    "class_b_platform_by_gender.csv"
)

save_platform_crosstab(
    OCCUPATION_COL,
    "class_b_platform_by_occupation.csv"
)

save_platform_crosstab(
    CITIZENSHIP_COL,
    "class_b_platform_by_citizenship.csv"
)

save_platform_crosstab(
    LANGUAGE_COL,
    "class_b_platform_by_language.csv"
)

In [14]:
df.to_csv(
    OUTPUT_DIR / "class_b_analysis_ready_features.csv",
    index=False,
    encoding="utf-8"
)

quality_summary = pd.DataFrame({
    "measure": [
        "rows",
        "unique_people",
        "unique_handles",
        "duplicate_handle_texts",
        "platforms"
    ],
    "value": [
        len(df),
        df[QID_COL].nunique(),
        df[HANDLE_COL].nunique(),
        df[HANDLE_COL].duplicated().sum(),
        df[PLATFORM_COL].nunique()
    ]
})

quality_summary.to_csv(
    OUTPUT_DIR / "class_b_dataset_quality_summary.csv",
    index=False
)

print("Analysis complete.")
print(quality_summary)

Analysis complete.
                  measure  value
0                    rows  20000
1           unique_people  19130
2          unique_handles  19750
3  duplicate_handle_texts    250
4               platforms      2
